# Kaggle — Unsloth LoRA 학습 (선택 경로)

- `finetune/train_lora_unsloth.py` + Add Data 로 연결한 `finetune_dataset.jsonl`.
- 표준 경로는 `kaggle_train.ipynb` → `train_lora.py`.
- **Internet On** 후 unsloth git 설치가 필요할 수 있음.

In [ ]:
import os

CODE_DATASET_SLUG = "metadata-code"
TRAIN_DATASET_SLUG = "metadata-train"

PROJECT_ROOT = f"/kaggle/input/{CODE_DATASET_SLUG}"
FINETUNE_DIR = f"{PROJECT_ROOT}/finetune"
DATA_PATH = f"/kaggle/input/{TRAIN_DATASET_SLUG}/finetune_dataset.jsonl"
OUTPUT_DIR = "/kaggle/working/lora-output-unsloth"
BASE_MODEL = "MLP-KTLim/llama-3-Korean-Bllossom-8B"

REPORT_TO = "none"
WANDB_RUN_NAME = "bllossom-sft-unsloth-kaggle"
PUSH_TO_HUB = False
HUB_MODEL_ID = ""

In [ ]:
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
from collections import deque
from pathlib import Path
import subprocess
import sys

script = Path(FINETUNE_DIR).resolve() / "train_lora_unsloth.py"
if not script.is_file():
    raise FileNotFoundError(f"train_lora_unsloth.py 없음: {script}")

cmd = [
    sys.executable,
    "-u",
    str(script),
    "--data", DATA_PATH,
    "--out", OUTPUT_DIR,
    "--base-model", BASE_MODEL,
    "--epochs", "3",
    "--batch-size", "2",
    "--grad-accum", "8",
    "--max-length", "2048",
    "--warmup-steps", "50",
    "--save-strategy", "steps",
    "--save-steps", "100",
    "--report-to", REPORT_TO,
    "--wandb-run-name", WANDB_RUN_NAME,
]
if PUSH_TO_HUB:
    cmd.append("--push-to-hub")
    cmd.extend(["--hub-model-id", HUB_MODEL_ID])

tail = deque(maxlen=400)
proc = subprocess.Popen(
    cmd,
    cwd=str(script.parent),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    tail.append(line)
    print(line, end="")
rc = proc.wait()
if rc != 0:
    raise RuntimeError("".join(tail))

print(f"학습 완료: {OUTPUT_DIR}")